# Metodologia Design Science Research (DSR)

**Etapa de Pesquisa (Peffers et al., 2007):**
### 3. Design e Desenvolvimento (Design and Development)

**Objetivo Acadêmico:** Este notebook consolida o "oceano de dados" coletado nos notebooks 01a a 01e em um único artefato analítico (Base Mestra). A unificação de bases heterogêneas (estruturadas e não estruturadas) exige um rigoroso processo de limpeza e alinhamento temporal, garantindo a integridade dos dados necessária para o ciclo de indução de modelos de IA, conforme preconizado por Hevner et al. (2004).


# 02 - Consolidação e Limpeza Rigorosa
Este notebook unifica todas as fontes de dados e aplica filtros estritos de qualidade:
1. Apenas dias letivos conforme o calendário.
2. Apenas dias com cardápio válido (proteína identificada).
3. Apenas dias com pelo menos 1 reserva e 1 consumo registrado.

In [1]:
import pandas as pd
import numpy as np
import os

# Configurações
DATA_DIR = '../data/'
ARQUIVOS = {
    'calendario': 'calendario_consolidado.csv',
    'clima': 'clima_consolidado.csv',
    'cardapio': 'cardapio_consolidado.csv',
    'consumo': 'consumo_anonimo.csv',
    'reservas': 'reservas_anonimas.csv'
}
SAIDA_MESTRA = os.path.join(DATA_DIR, 'base_mestra_consolidada.csv')

In [2]:
# 1. Carregar Datasets
dfs = {}
for nome, arq in ARQUIVOS.items():
    path = os.path.join(DATA_DIR, arq)
    if os.path.exists(path):
        dfs[nome] = pd.read_csv(path)
        dfs[nome]['data'] = pd.to_datetime(dfs[nome]['data'])
        print(f"✅ {nome.capitalize()} carregado: {len(dfs[nome])} linhas.")

✅ Calendario carregado: 365 linhas.
✅ Clima carregado: 1178 linhas.
✅ Cardapio carregado: 125 linhas.
✅ Consumo carregado: 23874 linhas.
✅ Reservas carregado: 19826 linhas.


In [3]:
# 2. Agregação de Reservas e Consumo
# Cruzamento Individual para fluxo
df_fluxo = pd.merge(
    dfs['reservas'][['data', 'id_aluno_anonimo']], 
    dfs['consumo'][['data', 'id_aluno_anonimo']], 
    on=['data', 'id_aluno_anonimo'], 
    how='outer', 
    indicator=True
)

metricas_diarias = df_fluxo.groupby('data')['_merge'].agg(
    reservou_e_comeu=lambda x: (x == 'both').sum(),
    reservou_e_nao_comeu=lambda x: (x == 'left_only').sum(),
    nao_reservou_e_comeu=lambda x: (x == 'right_only').sum()
).reset_index()

df_res_daily = dfs['reservas'].groupby('data').size().reset_index(name='total_reservas')
df_con_daily = dfs['consumo'].groupby('data').size().reset_index(name='total_servido')

df_consolidado_diario = pd.merge(df_res_daily, df_con_daily, on='data', how='outer')
df_consolidado_diario = pd.merge(df_consolidado_diario, metricas_diarias, on='data', how='outer').fillna(0)

In [4]:
# 3. Join Centralizado
df_mestra = dfs['calendario'].copy()
df_mestra = pd.merge(df_mestra, dfs['clima'], on='data', how='left')
df_mestra = pd.merge(df_mestra, dfs['cardapio'], on='data', how='left')
df_mestra = pd.merge(df_mestra, df_consolidado_diario, on='data', how='left')

print(f"🧩 Base Mestra inicial criada: {len(df_mestra)} linhas.")

🧩 Base Mestra inicial criada: 365 linhas.


In [5]:
# 4. Limpeza e FILTRAGEM RIGOROSA
cols_fluxo = ['total_reservas', 'total_servido', 'reservou_e_comeu', 'reservou_e_nao_comeu', 'nao_reservou_e_comeu']
df_mestra[cols_fluxo] = df_mestra[cols_fluxo].fillna(0).astype(int)

# Tratar nulos do cardápio e clima
cols_cat_cardapio = dfs['cardapio'].select_dtypes(include=['object', 'number']).columns.tolist()
if 'data' in cols_cat_cardapio: cols_cat_cardapio.remove('data')
df_mestra[cols_cat_cardapio] = df_mestra[cols_cat_cardapio].fillna('SEM_CONTEUDO')

# FILTROS ESTREITOS
mask_dados_completos = (
    (df_mestra['tem_refeicao'] == 1) & 
    (df_mestra['proteina_principal'] != 'SEM_CONTEUDO') &
    (df_mestra['total_reservas'] > 0) &
    (df_mestra['total_servido'] > 0)
)
df_final = df_mestra[mask_dados_completos].copy()

# 5. Exportação
df_final.to_csv(SAIDA_MESTRA, index=False)

print(f"✨ Consolidação concluída com FILTROS RÍGIDOS.")
print(f"📊 Dias úteis para modelagem (final): {len(df_final)}")
print(f"📅 Período: {df_final['data'].min().strftime('%d/%m/%Y')} até {df_final['data'].max().strftime('%d/%m/%Y')}")

✨ Consolidação concluída com FILTROS RÍGIDOS.
📊 Dias úteis para modelagem (final): 99
📅 Período: 24/02/2025 até 22/08/2025
